<div style="display: flex; align-items: center;">
  <img src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/DecisionIntelligenceWorkshopLogo.png" width="40px" style="margin-right: 10px;">
  <span style="font-size: 1.5em; font-weight: bold;">Workshop (AI Extensions) - Decisions with Reusable Prompts and Native .NET Functions</span>
</div>

Decision Intelligence applied in this module:  
* Listing various decision-making frameworks with their descriptions 
* Listing types of decision-making frameworks dynamically based on the decision type
* Reusing AI Extensions prompt templates with dynamic inputs
* Selecting a decision-making framework dynamically using native .NET code
* Decision Scenario: Using a Monte Carlo Simulation to estimate decision uncertainty

AI Extensions prompt templates are reusable instructions that help guide GenAI models toward a specific decision workflow. A prompt template usually has one responsibility: list useful decision frameworks, recommend a reasoning approach for a high-stakes decision, summarize a tradeoff, or personalize a recommendation from structured context.

What makes reusable prompt functions useful? GenAI models can already respond to basic prompts, but decision workflows usually need repeatable instructions, consistent formatting, and dynamic inputs. By combining Microsoft.Extensions.AI with simple C# helper methods, a prompt becomes a reusable part of an AI orchestration flow instead of a one-off string.

For example, imagine you want to prepare a great Thanksgiving dinner and you ask a GenAI cooking application to create a recipe. It may produce a delicious plan, but it may also use ingredients or cooking appliances you do not own. You could keep prompting until it narrows the recipe, but it is better to provide available ingredients, time availability, kitchen appliances, and allergy preferences as dynamic inputs. The same pattern applies to Decision Intelligence: reusable prompt templates let the model consider specific business context, user constraints, and decision criteria each time the prompt runs.

In this notebook, reusable prompt functions are implemented with:
* `IChatClient` from Microsoft.Extensions.AI
* `ChatMessage` roles for system and user instructions
* `ChatOptions` for model settings such as temperature, top-p, and maximum output tokens 
* C# raw string interpolation for dynamic prompt inputs

Below is an example of a reusable prompt template that can be used for facilitating decision-making. Note the `{decisionToMake}` input where the parameter can be dynamically passed in to provide additional specificity.

```csharp
var decisionPrompt = $"""
[DECISION FRAMEWORKS TO USE]
Price's Law
Pareto Principle
Laplace Rule of Succession
Eisenhower Matrix
Median Rule of Five
Second Order Thinking

[DECISION GUIDANCE]
Try to use the best fitting framework.  
Prefer using quantitative decision frameworks rather than qualitative ones.  
Use calculations or code-based validation when quantitative reasoning is required.

Use the decision frameworks above to help the user make the following decision:
{decisionToMake}
""";
```

The core value is that the prompt can be reused with different inputs while still flowing through the same AI Extensions `IChatClient` API.

----
### Step 1 - Initialize Configuration Builder & Build the AI Extensions Responses API Orchestration 

Execute the next two cells to:
* Use the Configuration Builder to load the API secrets.  
* Use the API configuration to build a Responses API-backed AI Extensions `IChatClient` orchestration.

In [1]:
#r "nuget: Microsoft.Extensions.Configuration, 10.0.8"
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.8"
#r "nuget: System.Text.Json, 10.0.8"

using Microsoft.Extensions.Configuration.Json;
using Microsoft.Extensions.Configuration;
using System.IO;
using System;

var configurationBuilder = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("local.settings.json", optional: true, reloadOnChange: true)
    .AddJsonFile("secrets.settings.json", optional: true, reloadOnChange: true);
var config = configurationBuilder.Build();

// IMPORTANT: You ONLY NEED either Azure OpenAI or OpenAI connection info, not both.
// Azure OpenAI Connection Info
var azureOpenAIEndpoint = config["AzureOpenAI:Endpoint"];
var azureOpenAIAPIKey = config["AzureOpenAI:APIKey"];
var azureOpenAIModelDeploymentName = config["AzureOpenAI:ModelDeploymentName"];
// OpenAI Connection Info 
var openAIAPIKey = config["OpenAI:APIKey"];
var openAIModelId = config["OpenAI:ModelId"];

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.Configuration, 10.0.8 Microsoft.Extensions.Configuration.Json, 10.0.8 System.Text.Json, 10.0.8

In [2]:
// Import the Microsoft Extensions AI NuGet Packages
#r "nuget: Microsoft.Extensions.AI, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.Abstractions, 10.6.0"
#r "nuget: Microsoft.Extensions.AI.OpenAI, 10.6.0"
// Import Azure & OpenAI NuGet Packages
#r "nuget: Azure.AI.OpenAI, 2.9.0-beta.1"
#r "nuget: Azure.Identity, 1.21.0"
#r "nuget: OpenAI, 2.10.0"

using Azure.AI.OpenAI;
using Microsoft.Extensions.AI;
using OpenAI;
using OpenAI.Responses;
using System.ClientModel;
using System.Collections.Generic;
using System.ComponentModel;
using System.Threading.Tasks;
using System.Text.Json;

// Set the flag to use Azure OpenAI or OpenAI. False to use OpenAI, True to use Azure OpenAI
var useAzureOpenAI = true;

// Create the IChatClient based on the selected service
IChatClient chatClient;

#pragma warning disable OPENAI001

if (useAzureOpenAI)
{
    Console.WriteLine("Using Azure OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(azureOpenAIAPIKey);
    var azureOpenAIClient = new AzureOpenAIClient(new Uri(azureOpenAIEndpoint), apiKeyCredential);

    // Get the ResponsesClient from AzureOpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = azureOpenAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: azureOpenAIModelDeploymentName);
}
else
{
    Console.WriteLine("Using OpenAI Service");

    var apiKeyCredential = new ApiKeyCredential(openAIAPIKey);
    var openAIClient = new OpenAIClient(apiKeyCredential);

    // Get the ResponsesClient from OpenAIClient and adapt it to Microsoft.Extensions.AI.
    var responsesClient = openAIClient.GetResponsesClient();
    chatClient = responsesClient.AsIChatClient(defaultModelId: openAIModelId);
}

#pragma warning restore OPENAI001

var decisionIntelligenceSystemPrompt = """
You are a Decision Intelligence assistant.
Help the user explore options, evaluate tradeoffs, reason through uncertainty, solve problems,
and apply systems thinking to personal, professional, strategic, and operational decisions.

Provide responses that are structured, logical, and thorough.
Aim to improve the user's judgment rather than make choices for them.
Be balanced, analytical, and pragmatic.
Adapt depth and complexity to the user's context.
""";

async Task<string> RunPromptAsync(string prompt, ChatOptions? options = null)
{
    ChatResponse response = await chatClient.GetResponseAsync(prompt, options);
    return response.Text ?? string.Empty;
}

async Task<string> RunDecisionPromptAsync(string userPrompt, ChatOptions? options = null)
{
    var chatMessages = new List<ChatMessage>
    {
        new(ChatRole.System, decisionIntelligenceSystemPrompt),
        new(ChatRole.User, userPrompt)
    };

    ChatResponse response = await chatClient.GetResponseAsync(chatMessages, options);
    return response.Text ?? string.Empty;
}

static string GetFunctionResultText(object? result)
{
    if (result is null)
    {
        return string.Empty;
    }

    if (result is JsonElement jsonElement)
    {
        return jsonElement.ValueKind == JsonValueKind.String
            ? jsonElement.GetString() ?? string.Empty
            : jsonElement.ToString();
    }

    return result.ToString() ?? string.Empty;
}


Installed Packages Azure.AI.OpenAI, 2.9.0-beta.1 Azure.Identity, 1.21.0 Microsoft.Extensions.AI, 10.6.0 Microsoft.Extensions.AI.Abstractions, 10.6.0 Microsoft.Extensions.AI.OpenAI, 10.6.0 OpenAI, 2.10.0

Using Azure OpenAI Service



(59,61): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(65,73): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

(77,43): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.

warning CS1701: Assuming assembly reference 'OpenAI, Version=2.9.1.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' used by 'Azure.AI.OpenAI' matches identity 'OpenAI, Version=2.10.0.0, Culture=neutral, PublicKeyToken=b4187f3e65366280' of 'OpenAI', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ClientModel, Version=1.9.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.AI.OpenAI' matches identity 'System.ClientModel, Version=1.10.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel'

----
### Step 2 - Create an AI Extensions Reusable Prompt Function 

> 📜 **_"Decision frameworks are like maps. Use them to navigate complex decision-making terrain._** 
>
> -- <cite>Unknown</cite> 

Reusable prompt functions in this notebook are simple C# helper methods that submit prompt templates through `IChatClient`. This keeps the prompt reusable while using the same AI Extensions API surface shown in the previous workshop notebooks.

In [3]:
// Simple prompt to list some decision frameworks this GenAI LLM is familiar with
var decisionFrameworksPromptTemplate = """
Identify and list various decision-making frameworks that can enhance the quality of decisions. 
Briefly describe how each framework supports better analysis and reasoning in different decision-making scenarios.

Output Format Instructions: 
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
""";

// Treat the prompt template as a reusable local function that invokes the AI Extensions IChatClient.
async Task<string> RunDecisionFrameworksPromptAsync(ChatOptions? options = null)
{
    return await RunDecisionPromptAsync(decisionFrameworksPromptTemplate, options);
}

var decisionFrameworksResponseText = await RunDecisionFrameworksPromptAsync();
decisionFrameworksResponseText.DisplayAs("text/markdown");

| Framework | What it is | How it improves decision quality | Best used for |
|---|---|---|---|
| Cost-Benefit Analysis (CBA) | Compares expected costs and benefits of an option, often in monetary terms. | Makes tradeoffs explicit and helps identify whether an action creates net value. | Investments, projects, policy choices, purchasing decisions. |
| Expected Value Analysis | Calculates the average outcome by weighting possible results by their probabilities. | Encourages probabilistic thinking and reduces reliance on intuition alone. | Decisions under uncertainty, finance, insurance, forecasting. |
| Decision Tree Analysis | Maps choices, chance events, and outcomes in a branching structure. | Clarifies sequences of decisions and helps compare complex paths systematically. | Multi-step decisions, clinical choices, product strategy, operations. |
| SWOT Analysis | Examines strengths, weaknesses, opportunities, and threats. | Provides a structured scan of internal and external factors to frame a decision. | Strategic planning, competitive analysis, organizational decisions. |
| Pareto Analysis (80/20 Rule) | Focuses on the small number of causes that drive most outcomes. | Helps prioritize efforts on the highest-impact issues first. | Problem-solving, quality improvement, resource allocation. |
| OODA Loop | Observe, Orient, Decide, Act; a rapid iterative decision cycle. | Improves adaptability by shortening feedback loops and enabling quick course correction. | Dynamic environments, crisis response, competitive strategy. |
| Cynefin Framework | Classifies situations as simple, complicated, complex, chaotic, or disorderly. | Matches the decision approach to the nature of the problem instead of forcing one method everywhere. | Ambiguous, uncertain, rapidly changing, or operational contexts. |
| Premortem Analysis | Assumes a decision has failed and asks why. | Surfaces hidden risks, blind spots, and failure modes before committing. | Project planning, strategy, high-stakes decisions. |
| Red Team / Devil’s Advocate | Intentionally challenges assumptions, plans, and conclusions. | Reduces groupthink and stress-tests reasoning by exposing weaknesses. | Security, policy, executive decisions, team planning. |
| Scenario Planning | Develops multiple plausible futures and tests decisions against them. | Improves resilience by preparing for uncertainty rather than predicting one outcome. | Long-term strategy, market shifts, geopolitical or technological change. |
| Multi-Criteria Decision Analysis (MCDA) | Scores options across several weighted criteria. | Makes subjective preferences more transparent and supports consistent comparison. | Vendor selection, hiring, portfolio choices, policy tradeoffs. |
| Analytical Hierarchy Process (AHP) | Breaks a decision into a hierarchy and uses pairwise comparisons to rank options. | Helps structure complex decisions and formalize preference judgments. | Procurement, prioritization, strategic resource allocation. |
| Rapid Evidence Assessment / Evidence-Based Decision Making | Uses the best available data and research before choosing. | Reduces bias and anchors decisions in credible information. | Healthcare, public policy, product, management decisions. |
| Bayesian Reasoning | Updates beliefs as new evidence arrives. | Supports continuous learning and prevents overconfidence in initial assumptions. | Diagnostics, forecasting, experimentation, risk management. |
| Expected Utility Theory | Chooses the option with the highest expected utility, not just highest payoff. | Accounts for preferences, risk tolerance, and non-monetary value. | High-stakes personal, financial, and strategic decisions. |
| Heuristic Checklists | Uses simple rules or checklists to guide consistent action. | Reduces omission errors and improves reliability in routine or high-pressure decisions. | Aviation, medicine, operations, quality control. |
| Six Thinking Hats | Separates thinking into modes such as facts, emotions, risks, benefits, creativity, and process. | Encourages balanced discussion and prevents one viewpoint from dominating. | Team decisions, ideation, conflict-prone meetings. |
| RCA / 5 Whys | Repeatedly asks “why” to trace a problem to its root cause. | Prevents treating symptoms only and helps design more durable fixes. | Troubleshooting, process improvement, incident analysis. |
| ZOPA / BATNA Analysis | Evaluates negotiating space and best alternatives to a negotiated agreement. | Improves leverage assessment and clarifies when to accept, reject, or negotiate. | Negotiations in business, employment, procurement, diplomacy. |
| Lean Startup / Build-Measure-Learn | Tests assumptions with small experiments and iterates based on feedback. | Reduces risk by validating decisions with real-world evidence quickly. | Product development, innovation, entrepreneurship. |


(17,64): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



----
### Step 3 - AI Extensions Dynamic Decision Intelligence  

> 📜 **_"If the only tool you have is a hammer, you tend to see every problem as a nail."_**
>
> -- <cite>Abraham Maslow (Renowned American psychologist)</cite> 

Prompt templates can be dynamically composed using C# variables and raw string interpolation. This allows the ease of passing in parameters and execution settings into reusable prompt functions. This is not groundbreaking, but it does allow you to dynamically compose AI prompts (context-engineering, prompt-engineering dynamically).

Execute the cell below to view how the prompt can dynamically instruct the LLM to retrieve different types of decision-making frameworks.

In [4]:
// Takes input variables and creates a dynamic prompt that can be used to invoke the GenAI model.
string CreateDecisionFrameworkPrompt(int numberOfFrameworks, string decisionType)
{
    return $"""
        Identify and list {numberOfFrameworks} decision-making frameworks that can enhance the quality of decisions. 
        Briefly describe how each framework supports better analysis and reasoning in {decisionType} decision-making scenarios.

        Output Format Instructions: 
        When generating Markdown, do not use any headings higher than ###. 
        Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
        All top-level section headers should start at ### or lower. 
        Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
        For separation, use extra extra spacing. Do not any render horizontal lines.
        Format the response using only a Markdown table. Only return a Markdown table. 
        Do not enclose the table in triple backticks.
        """;
}

// Create the chat options. 
//Try changing the settings to see how they affect the quality of generated text.
#pragma warning disable OPENAI001

var decisionFrameworkChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

#pragma warning restore OPENAI001

// Dynamically set the number of frameworks and decision type -> strategic
var strategicDecisionFrameworkPrompt = CreateDecisionFrameworkPrompt(
    numberOfFrameworks: 3,
    decisionType: "Long-Term strategic with probabilistic outcomes");

var strategicDecisionFrameworkResponseText = await RunDecisionPromptAsync(
    strategicDecisionFrameworkPrompt,
    decisionFrameworkChatOptions);

strategicDecisionFrameworkResponseText.DisplayAs("text/markdown");

| Framework | How it enhances long-term strategic, probabilistic decision-making |
|---|---|
| Expected Value / Expected Utility Analysis | Compares options by weighting each possible outcome by its probability and value, helping decision-makers choose the path with the best average long-run return rather than the most obvious short-term gain. It is useful when outcomes are uncertain and tradeoffs can be quantified. |
| Decision Tree Analysis | Structures complex decisions into sequences of choices, uncertainties, and outcomes. It improves reasoning by making assumptions explicit, showing how different paths unfold over time, and allowing probabilities and payoffs to be compared across scenarios. |
| Scenario Planning | Evaluates multiple plausible future environments instead of relying on a single forecast. It supports better long-term thinking by testing strategies against uncertainty, revealing robustness, and reducing overconfidence in any one prediction. |

In the example below, the number of frameworks to return has been changed and the type has been changed to **Quick to Implement for rapid Decision-Making**.

In [5]:
// Dynamically set the number of frameworks and decision type -> requiring fast action
var rapidDecisionFrameworkPrompt = CreateDecisionFrameworkPrompt(
    numberOfFrameworks: 5,
    decisionType: "Quick to Implement for rapid Decision-Making");

// Note: The invocation has NOT changed, only the dynamic prompt inputs have changed.
var rapidDecisionFrameworkResponseText = await RunDecisionPromptAsync(
    rapidDecisionFrameworkPrompt,
    decisionFrameworkChatOptions);

rapidDecisionFrameworkResponseText.DisplayAs("text/markdown");

| Framework | Core Idea | How It Supports Better Analysis and Reasoning in Rapid Decision-Making | Best Fit for Quick Implementation |
|---|---|---|---|
| OODA Loop (Observe–Orient–Decide–Act) | Repeatedly scan the situation, interpret it, choose an action, and learn from results. | Helps decision-makers stay agile by breaking fast-changing problems into a simple cycle. It reduces delay, encourages quick learning, and supports course correction when new information appears. | Time-sensitive situations with uncertainty, shifting conditions, or competitive pressure. |
| DECIDE Model | Define the problem, Establish criteria, Consider alternatives, Identify the best choice, Develop and implement, Evaluate results. | Provides a clear step-by-step structure that prevents rushed, unexamined choices. Even when used briefly, it improves reasoning by forcing clarity on the problem and a comparison of options. | Situations where a quick but structured decision is needed without deep analysis. |
| Eisenhower Matrix | Prioritize tasks by urgency and importance. | Improves judgment by separating what truly needs immediate action from what only feels urgent. This prevents reactive decision-making and helps focus effort on high-value choices. | Fast prioritization of tasks, requests, and operational decisions. |
| Cost-Benefit Analysis | Compare expected benefits and costs of each option. | Supports quick quantitative reasoning by making tradeoffs explicit. It helps avoid decisions based only on intuition and clarifies whether the likely upside justifies the effort or risk. | Simple choices where impacts can be estimated quickly, even if approximately. |
| Two-Way Door vs. One-Way Door Decisions | Distinguish reversible decisions from irreversible ones. | Improves speed and confidence by matching the level of analysis to the stakes. Reversible decisions can be made quickly, while irreversible ones get more scrutiny, reducing overthinking in low-risk cases. | Rapid decisions where some choices can be undone and others cannot. |

----
### Step 4 - Decision Scenario with Dynamic Decision Intelligence

In the below code a decision-analysis scenario is introduced that uses dynamic inputs to personalize the decision recommendation dynamically. The more specific information that provides contextual information to the Generative AI model can greatly improve the specific recommendations. 

**Decision Scenario:** Michael is a 35-year-old professional chef who is considering opening his own restaurant. This is a significant life decision that could greatly impact his career and personal life. Michael is looking for a recommendation for an approach (decision) for this potentially life-changing decision. 

<img style="display: block; margin: auto;" width ="700px" src="https://raw.githubusercontent.com/bartczernicki/DecisionIntelligence.GenAI.Workshop/main/Images/Scenarios/Scenario-RestaurantDecision.png">

Three factors will be considered for this decision scenario that the user will be prompted for: 
1) Michael's Total Net Worth in dollars (enter number)
2) Level of competition with other restaurants (low, medium, high) 
3) Level of support from Michael's friends and family (low, medium, high) 

**Various Decision Inputs:** You can simulate different What-If scenarios by varying the input of net worth, restaurant competition and level of family support. Note the different decision recommendations based on this provided by Artificial Intelligence. The recommendations in this example are quite simple, but even on the extreme inputs Generative AI has a concept of understanding the quality of inputs.

In [6]:
// Import the Microsoft.DotNet.Interactive namespace for user input
using Microsoft.DotNet.Interactive;

var totalNetWorth = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Michael's total net worth in dollars: ");
var levelOfCompetition = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Level of competition with other restaurants (Low, Medium, High): ");
var levelOfFamilySupport = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Level of support from Michael's friends and family (Low, Medium, High): ");

Console.WriteLine($"Michael's Net Worth: {totalNetWorth}");
Console.WriteLine($"Level of Restaurant Competition: {levelOfCompetition}");
Console.WriteLine($"Michael's level of Family Support: {levelOfFamilySupport}");
Console.WriteLine();

// Takes dynamic inputs and creates a personalized prompt that can be used to invoke the model.
var restaurantDecisionRecommendationPrompt = $"""
Michael is a 35-year-old professional chef who is considering opening his own restaurant. 
This is a significant life decision that could greatly impact his career and personal life. 
Michael is looking for a recommendation on how to approach this.
Some key information about Michael:
- Michael's total net worth is (in US dollars) ${totalNetWorth}.
- The level of competition with other restaurants is {levelOfCompetition}.
- The level of support from Michael's friends and family is {levelOfFamilySupport}.

Based on this information, what recommendation would you give to Michael regarding opening his own restaurant?

Output Format Instructions: 
When generating Markdown, do not use any headings higher than ###. 
Avoid # and ## headers. Use only ###, ####, or lower-level headings if necessary. 
All top-level section headers should start at ### or lower. 
Never use ---, ***, or ___ for horizontal lines. There should be no horizontal lines in the output.
For separation, use extra extra spacing. Do not any render horizontal lines.
Format the response using only a Markdown table. Only return a Markdown table. 
Do not enclose the table in triple backticks.
Provide a small Decision Summary of the recommendation below the table.
""";


#pragma warning disable OPENAI001

// Create the chat options. Try changing the settings to see how they affect the quality of generated text.
var restaurantDecisionChatOptions = new ChatOptions
{
    RawRepresentationFactory = _ => new OpenAI.Responses.CreateResponseOptions
    {
        Model = useAzureOpenAI ? azureOpenAIModelDeploymentName : openAIModelId,
        ReasoningOptions = new OpenAI.Responses.ResponseReasoningOptions
        {
            ReasoningEffortLevel = OpenAI.Responses.ResponseReasoningEffortLevel.Medium,
            ReasoningSummaryVerbosity = OpenAI.Responses.ResponseReasoningSummaryVerbosity.Detailed
        },
        StoredOutputEnabled = false
    }
};

var restaurantDecisionRecommendationResponseText = await RunDecisionPromptAsync(
    restaurantDecisionRecommendationPrompt,
    restaurantDecisionChatOptions);

restaurantDecisionRecommendationResponseText.DisplayAs("text/markdown");

#pragma warning restore OPENAI001

Michael's Net Worth: 1000000
Level of Restaurant Competition: Medium
Michael's level of Family Support: Medium



| Aspect | Assessment | Recommendation |
|---|---|---|
| Overall recommendation | Michael has the financial capacity to take a measured entrepreneurial risk, but the business environment is not low-risk. | **Proceed cautiously, not impulsively.** Open the restaurant only after validating the concept and building a strong risk-management plan. |
| Financial position | With a net worth of **$1,000,000**, he likely has enough cushion to fund a well-structured launch without endangering his entire financial life. | Risk a **limited, pre-defined portion** of his net worth rather than overcommitting. Preserve personal liquidity and an emergency reserve. |
| Market conditions | **Medium competition** suggests the restaurant will need a clear differentiator to succeed. | Enter only if he can offer a distinct concept, strong location strategy, and disciplined cost control. |
| Personal support | **Medium support** from friends and family is helpful but not a strong safety net. | Avoid relying on informal support for capital, labor, or emotional backup; build professional support around him instead. |
| Best approach | A full-scale launch increases downside if the concept misses. | Consider a **phased entry**: pop-up, food truck, tasting events, or a smaller first location before scaling up. |
| Decision Summary | Michael is in a position where opening a restaurant could be a reasonable career move, but the safest path is to test the concept first and structure the business to limit downside. | **Recommendation: Yes, but only with a phased, well-capitalized, and carefully validated plan.** |

----
### Native .NET Functions with Microsoft.Extensions.AI

The first part of this notebook used reusable prompt templates. The next section adds native .NET functions with `AIFunctionFactory`, which lets AI workflows use deterministic C# logic for business rules, calculations, simulations, API calls, and other capabilities that should not be left to prompting alone.

### Step 5 - Create a Simple Native .NET Function with AI Extensions

Execute the cell below to create a very simple native .NET function using a C# inline method. Notice the function takes no parameters. It retrieves the name of a productivity decision framework to use. In this case, that return name is hard-coded to "Price's Law".

The function returns instantly because it is not calling a GenAI service. The point is to show how Microsoft.Extensions.AI can wrap deterministic .NET logic as a first-class `AIFunction`.

In [7]:
// Create an AIFunction from an inline C# method.
var nameOfProductivityFramework = AIFunctionFactory.Create(
    (Func<string>)(() => "Price's Law"),
    new AIFunctionFactoryOptions
    {
        Name = "GetNameOfProductivityFramework",
        Description = "Retrieves the name of the Productivity Framework to use."
    });

// Invoke the function directly with AI Extensions.
var response = await nameOfProductivityFramework.InvokeAsync(new AIFunctionArguments());
Console.WriteLine(GetFunctionResultText(response));

Price's Law


### Step 6 - Create a Native .NET Function with Dynamic Parameters

Execute the cell below to create a native .NET function that takes a parameter as input. This shows that C# native functions can have different execution paths. The execution paths can obviously be quite complex. Basically, any C# logic flow will work.

In [8]:
// Create an AIFunction from an inline C# method with parameters.
var nameOfProductivityFrameworkByType = AIFunctionFactory.Create(
    (Func<string, string>)(typeOfProductivity => typeOfProductivity == "Sales" ? "Price's Law" : "Pareto Principle"),
    new AIFunctionFactoryOptions
    {
        Name = "GetNameOfProductivityFramework",
        Description = "Retrieves the name of the Productivity Framework to use."
    });

// Pass the "Sales" parameter to the function.
var salesResponse = await nameOfProductivityFrameworkByType.InvokeAsync(new AIFunctionArguments
{
    ["typeOfProductivity"] = "Sales"
});
Console.WriteLine(GetFunctionResultText(salesResponse));

// Pass the "Other" parameter to the function.
var otherResponse = await nameOfProductivityFrameworkByType.InvokeAsync(new AIFunctionArguments
{
    ["typeOfProductivity"] = "Other"
});
Console.WriteLine(GetFunctionResultText(otherResponse));

Price's Law
Pareto Principle


### Step 7 - Use Native .NET Functions To Simulate the Uncertainty of a Decision  

> 📜 **_"The best way to predict the future is to simulate it. And the best way to simulate it is with Monte Carlo."_**
>
> -- <cite>Nassim Nicholas Taleb (Lebanese-American essayist, scholar, best known for his work on probability)</cite>

For more complex decisions, native .NET functions can use statistics, advanced probabilistic algorithms, analytics, machine learning, AI, and other approaches that have been relied on in software for decades. One such method is Monte Carlo Simulation. These powerful Monte Carlo simulation techniques are used everywhere: risk management, sports gambling, medical decision-making, game theory, energy market forecasting, and more. In simple terms, a Monte Carlo simulation is a series of many runs testing different plausible parameters. Running a Monte Carlo simulation many times results in an output of a plausible range.  

A simple use case for a Monte Carlo simulation is to provide a realistic range for an average probability. Imagine you want to illustrate the uncertainty of a decision that you have calculated to be 70% successful. On "average" the probability of success can be interpreted as 70%. However, what is the range of possible successes if that 70% decision model is run 100x? A Monte Carlo simulation can help solve that answer.

Run the cell below to create a new native .NET function that will take in the confidence parameter and output a simple string with the lower and upper bounds of a 95% confidence interval. A 95% Confidence Interval output will tell us if we execute this calculated decision 100x that has a 70% success probability, what is the realistic range of success in those 100 decisions.

In [9]:
string RetrieveConfidenceIntervalMonteCarlo(
    [Description("Claimed Probability Percentage")] int probability)
{
    const int NUMBEROFSIMULATIONS = 100000; // 100,000 simulations, make this smaller for faster results
    Console.WriteLine($"Simulating {NUMBEROFSIMULATIONS:n0} iterations with a claimed decision confidence of {probability}%...");

    var random = new Random(); // Add seed for reproducibility
    var bootstrapConfidenceScores = new List<double>();
    for (int i = 0; i != NUMBEROFSIMULATIONS; i++) // Bootstrap Simulations (bootstrap estimates)
    {
        var bootstrapSample = new List<double>();
        for (int j = 0; j != 100; j++)
        {
            var randomIndex = random.Next(0, 100);

            if (randomIndex < probability)
            {
                bootstrapSample.Add(1);
            }
        }

        bootstrapConfidenceScores.Add(bootstrapSample.Count);
    }

    // Sort the confidence scores to calculate the percentiles
    var bootstrapConfidenceScoresSorted = bootstrapConfidenceScores.OrderBy(a => a).ToList();
    // Calculate the 2.5% and 97.5% percentiles
    var lowerPercentileIndex = Convert.ToInt32(0.025 * NUMBEROFSIMULATIONS);
    var topPercentileIndex = Convert.ToInt32(0.975 * NUMBEROFSIMULATIONS);

    var lowerPercentile = Math.Round(bootstrapConfidenceScoresSorted[lowerPercentileIndex], 3);
    var upperPercentile = Math.Round(bootstrapConfidenceScoresSorted[topPercentileIndex], 3);

    var confidenceUncertaintyRange = $"95% Confidence Interval of a {probability}% success model: {lowerPercentile} to {upperPercentile} successful decision outcomes of 100 decisions made.";
    return confidenceUncertaintyRange;
}

Run the cell below to wrap the native .NET method as an `AIFunction` and invoke it directly. This will run a Monte Carlo Simulation with 100,000 simulations of a decision with a confidence (probability) of 70% being run 100x. Basically, this will provide the uncertainty range if you had made 100 decisions of the same 70% success decision.

In [10]:
var retrieveConfidenceInterval = AIFunctionFactory.Create(
    (Func<int, string>)RetrieveConfidenceIntervalMonteCarlo,
    new AIFunctionFactoryOptions
    {
        Name = "RetrieveConfidenceIntervalMonteCarlo",
        Description = "Generates Confidence Interval from provided Confidence Percentage"
    });

// Change the probability to another integer and invoke the function, to see other Confidence Intervals.
var confidenceIntervalRange70 = await retrieveConfidenceInterval.InvokeAsync(new AIFunctionArguments
{
    ["probability"] = 70
});
Console.WriteLine(GetFunctionResultText(confidenceIntervalRange70));

Simulating 100,000 iterations with a claimed decision confidence of 70%...
95% Confidence Interval of a 70% success model: 61 to 79 successful decision outcomes of 100 decisions made.
